In [ ]:
!df -h /content


Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   71G  38% /


In [ ]:
import requests

url = "https://s3.amazonaws.com/drivendata-public-assets/stac.tar"

r = requests.head(url, allow_redirects=True, timeout=30)

print("HTTP status:", r.status_code)
print("File size (bytes):", r.headers.get("Content-Length"))
print("File type:", r.headers.get("Content-Type"))

HTTP status: 200
File size (bytes): 33756133376
File type: application/x-tar


In [ ]:
import requests
from tqdm.auto import tqdm

url = "https://s3.amazonaws.com/drivendata-public-assets/stac.tar"
output_path = "/content/stac.tar"

with requests.get(url, stream=True, timeout=60) as response:
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))

    with open(output_path, "wb") as f, tqdm(
        total=total, unit="B", unit_scale=True, desc="Downloading"
    ) as progress:
        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                f.write(chunk)
                progress.update(len(chunk))

print("Download completed:", output_path)

Downloading:   0%|          | 0.00/33.8G [00:00<?, ?B/s]

Download completed: /content/stac.tar


In [ ]:
!ls -lh /content/stac.tar
!df -h /content

-rw-r--r-- 1 root root 32G Oct 20 09:36 /content/stac.tar
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   74G   39G  66% /


In [ ]:
!tar -tf /content/stac.tar | head -40

stac/
stac/catalog.json
stac/colombia/
stac/guatemala/
stac/st_lucia/
stac/st_lucia/castries/
stac/st_lucia/catalog.json
stac/st_lucia/dennery/
stac/st_lucia/gros_islet/
stac/st_lucia/gros_islet/collection.json
stac/st_lucia/gros_islet/gros_islet-imagery.json
stac/st_lucia/gros_islet/gros_islet-labels-train.json
stac/st_lucia/gros_islet/gros_islet_ortho-cog-thumbnail.png
tar: Ignoring unknown extended header keyword 'SCHILY.dev'
tar: Ignoring unknown extended header keyword 'SCHILY.ino'
tar: Ignoring unknown extended header keyword 'SCHILY.nlink'
stac/st_lucia/gros_islet/gros_islet_ortho-cog.tif
stac/st_lucia/gros_islet/train-gros_islet.geojson
stac/st_lucia/dennery/collection.json
stac/st_lucia/dennery/dennery-imagery.json
stac/st_lucia/dennery/dennery-labels-test.json
stac/st_lucia/dennery/dennery-labels-train.json
stac/st_lucia/dennery/dennery_ortho-cog-thumbnail.png
stac/st_lucia/dennery/dennery_ortho-cog.tif
stac/st_lucia/dennery/test-dennery.geojson
stac/st_lucia/dennery/train-de

In [ ]:
!tar -tvf /content/stac.tar | head -20

tar: Ignoring unknown extended header keyword 'SCHILY.dev'
tar: Ignoring unknown extended header keyword 'SCHILY.ino'
tar: Ignoring unknown extended header keyword 'SCHILY.nlink'
drwxr-xr-x braeburn/staff    0 2019-09-13 19:10 stac/
-rw-r--r-- braeburn/staff  646 2019-09-13 18:30 stac/catalog.json
drwxr-xr-x braeburn/staff    0 2019-09-13 19:10 stac/colombia/
drwxr-xr-x braeburn/staff    0 2019-09-13 19:15 stac/guatemala/
drwxr-xr-x braeburn/staff    0 2019-09-13 19:15 stac/st_lucia/
drwxr-xr-x braeburn/staff    0 2019-09-13 19:30 stac/st_lucia/castries/
-rw-r--r-- braeburn/staff  811 2019-09-13 18:35 stac/st_lucia/catalog.json
drwxr-xr-x braeburn/staff    0 2019-09-13 19:23 stac/st_lucia/dennery/
drwxr-xr-x braeburn/staff    0 2019-09-13 19:32 stac/st_lucia/gros_islet/
-rw-r--r-- braeburn/staff 1617 2019-09-13 18:37 stac/st_lucia/gros_islet/collection.json
-rw-r--r-- braeburn/staff 29652238 2019-09-13 18:37 stac/st_lucia/gros_islet/gros_islet-imagery.json
-rw-r--r-- braeburn/staff    

In [ ]:
!tar -tvf /content/stac.tar | grep 'stac/colombia/borde_rural/'

tar: Ignoring unknown extended header keyword 'SCHILY.dev'
tar: Ignoring unknown extended header keyword 'SCHILY.ino'
tar: Ignoring unknown extended header keyword 'SCHILY.nlink'
drwxr-xr-x braeburn/staff          0 2019-09-13 19:17 stac/colombia/borde_rural/
-rw-r--r-- braeburn/staff   14082919 2019-09-13 18:30 stac/colombia/borde_rural/borde_rural-imagery.json
-rw-r--r-- braeburn/staff       2267 2019-09-13 18:30 stac/colombia/borde_rural/borde_rural-labels-test.json
-rw-r--r-- braeburn/staff       2958 2019-09-13 18:30 stac/colombia/borde_rural/borde_rural-labels-train.json
-rw-r--r-- braeburn/staff    2654381 2019-09-13 18:30 stac/colombia/borde_rural/borde_rural_ortho-cog-thumbnail.png
-rw-r--r-- braeburn/staff 4471229119 2019-09-13 18:30 stac/colombia/borde_rural/borde_rural_ortho-cog.tif
-rw-r--r-- braeburn/staff       1772 2019-09-13 18:30 stac/colombia/borde_rural/collection.json
-rw-r--r-- braeburn/staff    1554297 2019-09-13 18:30 stac/colombia/borde_rural/test-borde_rural.g

In [ ]:
import tarfile
import os

archive_path = "/content/stac.tar"
output_dir = "/content/roof_material_data"

wanted_files = [
    "stac/colombia/borde_rural/borde_rural_ortho-cog.tif",
    "stac/colombia/borde_rural/train-borde_rural.geojson",
    "stac/colombia/borde_rural/test-borde_rural.geojson",
    "stac/colombia/borde_rural/borde_rural_ortho-cog-thumbnail.png",
    "stac/colombia/borde_rural/borde_rural-imagery.json",
    "stac/colombia/borde_rural/borde_rural-labels-train.json",
    "stac/colombia/borde_rural/borde_rural-labels-test.json",
    "stac/colombia/borde_rural/collection.json",
]

os.makedirs(output_dir, exist_ok=True)

with tarfile.open(archive_path, "r:") as tar:
    members = [
        m for m in tar.getmembers()
        if m.name in wanted_files
    ]
    tar.extractall(path=output_dir, members=members, filter="data")

print("Extracted files:")
for m in members:
    print(m.name, f"({m.size / (1024**3):.2f} GiB)")

Extracted files:
stac/colombia/borde_rural/borde_rural-imagery.json (0.01 GiB)
stac/colombia/borde_rural/borde_rural-labels-test.json (0.00 GiB)
stac/colombia/borde_rural/borde_rural-labels-train.json (0.00 GiB)
stac/colombia/borde_rural/borde_rural_ortho-cog-thumbnail.png (0.00 GiB)
stac/colombia/borde_rural/borde_rural_ortho-cog.tif (4.16 GiB)
stac/colombia/borde_rural/collection.json (0.00 GiB)
stac/colombia/borde_rural/test-borde_rural.geojson (0.00 GiB)
stac/colombia/borde_rural/train-borde_rural.geojson (0.00 GiB)


In [ ]:
!ls -lh /content/roof_material_data/stac/colombia/borde_rural/
!df -h /content

total 4.2G
-rw-r--r-- 1 root root  14M Sep 13  2019 borde_rural-imagery.json
-rw-r--r-- 1 root root 2.3K Sep 13  2019 borde_rural-labels-test.json
-rw-r--r-- 1 root root 2.9K Sep 13  2019 borde_rural-labels-train.json
-rw-r--r-- 1 root root 2.6M Sep 13  2019 borde_rural_ortho-cog-thumbnail.png
-rw-r--r-- 1 root root 4.2G Sep 13  2019 borde_rural_ortho-cog.tif
-rw-r--r-- 1 root root 1.8K Sep 13  2019 collection.json
-rw-r--r-- 1 root root 1.5M Sep 13  2019 test-borde_rural.geojson
-rw-r--r-- 1 root root 3.3M Sep 13  2019 train-borde_rural.geojson
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   78G   35G  70% /


In [ ]:
!pip -q install rasterio geopandas

In [ ]:
import rasterio
import geopandas as gpd
from pathlib import Path

folder = Path("/content/roof_material_data/stac/colombia/borde_rural")

image_path = folder / "borde_rural_ortho-cog.tif"
train_path = folder / "train-borde_rural.geojson"
test_path = folder / "test-borde_rural.geojson"

# Inspect the aerial image
with rasterio.open(image_path) as src:
    print("Image dimensions:", src.width, "x", src.height)
    print("Number of bands:", src.count)
    print("Coordinate reference system:", src.crs)
    print("Image bounds:", src.bounds)

# Inspect the labels
train = gpd.read_file(train_path)
test = gpd.read_file(test_path)

print("\nTraining buildings:", len(train))
print("Test buildings:", len(test))
print("\nTraining columns:", train.columns.tolist())
print("\nRoof-material counts:")
print(train["roof_material"].value_counts(dropna=False))

Image dimensions: 31315 x 52318
Number of bands: 4
Coordinate reference system: EPSG:32618
Image bounds: BoundingBox(left=592371.56518, bottom=501705.04172000004, right=593606.62878, top=503768.46364000003)

Training buildings: 4361
Test buildings: 2160

Training columns: ['id', 'roof_material', 'verified', 'geometry']

Roof-material counts:
roof_material
healthy_metal      2383
irregular_metal    1508
incomplete          245
concrete_cement     217
other                 8
Name: count, dtype: int64


In [ ]:
import rasterio
import numpy as np
from pathlib import Path

image_path = Path(
    "/content/roof_material_data/stac/colombia/"
    "borde_rural/borde_rural_ortho-cog.tif"
)

with rasterio.open(image_path) as src:
    print("Band descriptions:", src.descriptions)
    print("Data type:", src.dtypes)
    print("No-data value:", src.nodata)

    for i in range(1, src.count + 1):
        band = src.read(i, out_shape=(1, 1000, 1000))
        print(
            f"Band {i}: min={band.min()}, "
            f"max={band.max()}, mean={band.mean():.2f}"
        )

Band descriptions: (None, None, None, None)
Data type: ('uint8', 'uint8', 'uint8', 'uint8')
No-data value: None
Band 1: min=0, max=255, mean=83.88
Band 2: min=0, max=254, mean=82.15
Band 3: min=0, max=255, mean=66.70
Band 4: min=0, max=255, mean=171.00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p "/content/drive/MyDrive/RoofMaterialProject"

!cp -r \
"/content/roof_material_data/stac/colombia/borde_rural" \
"/content/drive/MyDrive/RoofMaterialProject/"

In [ ]:
!ls -lh "/content/drive/MyDrive/RoofMaterialProject/borde_rural"

total 4.2G
-rw------- 1 root root  14M Oct 20 09:52 borde_rural-imagery.json
-rw------- 1 root root 2.3K Oct 20 09:52 borde_rural-labels-test.json
-rw------- 1 root root 2.9K Oct 20 09:52 borde_rural-labels-train.json
-rw------- 1 root root 2.6M Oct 20 09:52 borde_rural_ortho-cog-thumbnail.png
-rw------- 1 root root 4.2G Oct 20 09:52 borde_rural_ortho-cog.tif
-rw------- 1 root root 1.8K Oct 20 09:52 collection.json
-rw------- 1 root root 1.5M Oct 20 09:52 test-borde_rural.geojson
-rw------- 1 root root 3.3M Oct 20 09:52 train-borde_rural.geojson


In [ ]:
!mkdir -p /content/roof_material_data/stac/colombia

!cp -r \
"/content/drive/MyDrive/RoofMaterialProject/borde_rural" \
"/content/roof_material_data/stac/colombia/"

In [ ]:
import os

folder = "/content/drive/MyDrive/RoofMaterialProject/borde_rural"

for name in os.listdir(folder):
    path = os.path.join(folder, name)
    print(f"{name}: {os.path.getsize(path) / (1024**3):.2f} GiB")

borde_rural-imagery.json: 0.01 GiB
borde_rural-labels-test.json: 0.00 GiB
borde_rural-labels-train.json: 0.00 GiB
borde_rural_ortho-cog-thumbnail.png: 0.00 GiB
borde_rural_ortho-cog.tif: 4.16 GiB
collection.json: 0.00 GiB
test-borde_rural.geojson: 0.00 GiB
train-borde_rural.geojson: 0.00 GiB


In [ ]:
import os

print("Original archive exists:", os.path.exists("/content/stac.tar"))
print("Borde Rural saved in Drive:",
      os.path.exists("/content/drive/MyDrive/RoofMaterialProject/borde_rural"))

Original archive exists: False
Borde Rural saved in Drive: False
